# SLC Partner Lab — magnetization of a 1D Ising chain

> This is the guided, graded exercise of the Fall Fest 2026 error-mitigation track. The open hackathon challenge is **Noise Busters** (`noise_busters.ipynb`), which uses the same tools with no fill-in-the-blanks.

Shaded lightcones (SLC) cut the sampling cost of [probabilistic error cancellation (PEC)](https://quantum.cloud.ibm.com/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec). PEC learns a sparse [Pauli–Lindblad](https://arxiv.org/abs/2201.09866) noise model for each unique circuit layer and cancels the noise, rescaling the result by $\gamma = \exp(\sum_{l,\sigma} 2\lambda_{l,\sigma})$, which multiplies the required shots by $\gamma^2$. SLC bounds, with cheap classical simulations, how much each error can affect the measurement: forward bounds propagate an error to the end of the circuit and bound its commutator with the observable, backward bounds propagate it to the start and bound its commutator with the initial state. Only the errors that matter get mitigated, within a bias tolerance you choose, trading QPU time for CPU time. For the theory, see the [SLC tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/pec-with-shaded-lightcones), [arXiv:2409.04401](https://arxiv.org/abs/2409.04401), and QGSS 2026 [Lab 3](https://github.com/qiskit-community/qgss-2026/tree/main/lab-3), currently the only other step-by-step walkthrough of this workflow. No prior experience with these tools is assumed.

Here you evolve a 20-qubit Ising chain forward in time and mitigate the magnetization profile $\langle Z_i\rangle$ at every site. At 20 qubits an exact $2^{20}$ statevector reference is still cheap to compute, so the result can be scored against it.

### How the pieces fit

The runtime tools used here are new, so here is the whole pipeline in five lines:

1. **Boxing** (Samplomatic's `generate_boxing_pass_manager`) groups the circuit into layers ("boxes") and annotates each one with twirling and noise-injection instructions.
2. **`NoiseLearnerV3`** learns a sparse Pauli–Lindblad noise model for each *unique* 2-qubit layer. Repeated layers reuse the same model.
3. **SLC** (this add-on) decides, per observable, which learned noise terms are worth cancelling, using cheap classical bounds.
4. **Samplomatic** is the sampling layer: `samplomatic.build` turns the boxed circuit into a template circuit plus a *samplex*, and the samplex draws a fresh randomization for every execution: the Pauli twirls and the anti-noise Paulis sampled from the learned model. That sampling is what implements PEC.
5. The **`Executor`** runs the randomized instances, and post-processing (readout correction, post-selection, $\gamma$ rescaling) turns the bitstrings into $\langle Z_i\rangle$.

__QPU budget:__ noise learning takes about 3 minutes of QPU time and the Executor job adds a few more. On the Open Plan your monthly allowance is small, so check your remaining time before submitting, and reuse finished job IDs instead of resubmitting.

## Step 0. Setup

Install dependencies (comment the install cell out after the first run), then import everything.

In [ ]:
%pip install -U qiskit qiskit-ibm-runtime
%pip install -U qiskit-addon-slc qiskit-addon-utils samplomatic
%pip install -U "qiskit[visualization]" pylatexenc
%pip install -U qc-grader

In [ ]:
import numpy as np
import samplomatic
from matplotlib import pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler import PassManager, generate_preset_pass_manager

from qiskit_addon_slc.bounds import (
    compute_backward_bounds,
    compute_forward_bounds,
    compute_local_scales,
    merge_bounds,
    tighten_with_speed_limit,
)
from qiskit_addon_slc.utils import generate_noise_model_paulis, map_modifier_ref_to_ref
from qiskit_addon_slc.visualization import draw_shaded_lightcone

from qiskit_addon_utils.exp_vals.expectation_values import executor_expectation_values
from qiskit_addon_utils.exp_vals.measurement_bases import get_measurement_bases
from qiskit_addon_utils.exp_vals.observable_mappings import map_observable_virtual_to_canonical
from qiskit_addon_utils.noise_management import gamma_from_noisy_boxes, trex_factors
from qiskit_addon_utils.noise_management.post_selection import PostSelector
from qiskit_addon_utils.noise_management.post_selection.transpiler.passes import (
    AddPostSelectionMeasures,
    AddSpectatorMeasures,
)

from qiskit_ibm_runtime import Executor, QiskitRuntimeService, QuantumProgram
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions

from qc_grader.challenges.fallfest_2026.magnetization import (
    grade_magnetization_ex1,
    grade_magnetization_ex2,
    grade_magnetization_ex3,
    grade_magnetization_ex4,
)


## Step 1. Map the problem

We evolve a 1D transverse-field Ising chain with a Trotterized kicked-Ising circuit: an $R_x(\theta)$ kick on every qubit, then even and odd `(sdg, cz)` bond layers (equivalent to $R_{zz}(-\pi/2)$) in a brick-wall pattern. `rx_angle = π/4` is deliberately non-Clifford. A non-Clifford kick keeps the fractional shading non-trivial, which is what this lab illustrates. We measure every qubit in the computational basis; the profile is $M_i = \langle Z_i\rangle$ with mean $\bar M = \tfrac{1}{N}\sum_i \langle Z_i\rangle$. The mitigated profile uses one program item per site, and a single unmitigated item will provide the raw baseline for all sites.

> **New to the Ising model?** You don't need physics background for this. The Ising model is the generic model for binary (0/1) variables with pairwise interactions, and it shows up well beyond physics. The transverse-field version used here just adds a single-qubit $R_x$ rotation layer between the $ZZ$ interaction layers. Treat it as the benchmark circuit: what this lab is about is the noise mitigation, not the model.

The circuit and observables below are **fixed**. Leave them as they are.

In [ ]:
num_qubits = 20
num_trotter_steps = 4
rx_angle = np.pi / 4   # non-Clifford: keeps the shaded-lightcone visualization meaningful


def construct_ising_circuit(num_qubits, num_trotter_steps, rx_angle, barrier=True):
    # Reused tutorial circuit: an rx layer + (sdg, cz) layers equivalent to Rzz(-pi/2).
    circuit = QuantumCircuit(num_qubits)
    for _step in range(num_trotter_steps):
        circuit.rx(rx_angle, range(num_qubits))
        if barrier:
            circuit.barrier()
        for first_qubit in (1, 2):
            for idx in range(first_qubit, num_qubits, 2):
                circuit.sdg([idx - 1, idx])
                circuit.cz(idx - 1, idx)
        if barrier:
            circuit.barrier()
    return circuit


# forward evolution + measure all qubits (extract every Z_i from the same bitstrings)
circuit = construct_ising_circuit(num_qubits, num_trotter_steps, rx_angle, barrier=False)
circuit.measure_active()
circuit.draw("mpl", fold=-1, scale=0.4, idle_wires=False, measure_arrows=False)


Define one single-Pauli observable $Z_i$ per site. SLC handles one observable at a time, which is why later steps run once per site.

In [ ]:
# magnetization profile: M_i = <Z_i>,  mean M = (1/N) * sum_i <Z_i>
site_observables = [
    SparsePauliOp.from_sparse_list([("Z", [i], 1.0)], num_qubits=num_qubits)
    for i in range(num_qubits)
]

print(f"number of site observables : {len(site_observables)}")
for i, obs in enumerate(site_observables):
    print(f"  Z_{i} : {obs.paulis[0]}")


### Pick a backend and lock in a qubit chain

Take the least busy operational Heron backend and reopen it with `use_fractional_gates=True`,
as the tutorial does: the native `rx` gate supports the non-Clifford kicks and is required by
the post-selection `x_pulse_type="rx"`. Then let the transpiler pick a chain at
`optimization_level=3` and re-transpile onto that fixed layout at `optimization_level=0`,
which keeps the `(sdg, cz)` layer structure, and with it the noise model, aligned with the
boxes.

In [ ]:
# --- replace with your own credentials if needed ---
service = QiskitRuntimeService()  # uses a saved account (or pass channel/token/instance)
backend = service.least_busy(
    operational=True,
    simulator=False,
    filters=lambda b: b.processor_type.get('family') == 'Heron'
)
# Reopen with fractional gates, as in the tutorial: the post-selection x-pulse
# (x_pulse_type="rx") must be one of the backend's basis gates.
backend = service.backend(backend.name, use_fractional_gates=True)

print("backend:", backend.name)


In [ ]:
# Pick the qubit chain automatically, then keep the layer structure intact.
# Pass 1: optimization_level=3 chooses a good chain for this backend.
layout_pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
layout = layout_pm.run(circuit).layout.final_index_layout()

print("transpiler-selected physical qubits:", layout)

# Pass 2: transpile onto that fixed layout with optimization_level=0, so the (sdg, cz)
# layer structure is preserved and the noise model still lines up with the boxes.
isa_pm = generate_preset_pass_manager(backend=backend, initial_layout=layout,
                                      optimization_level=0)
isa_circuit = isa_pm.run(circuit)
assert isa_circuit.layout.final_index_layout() == layout

# wire order for the shaded-lightcone plots: our chain first, spectators after
wire_order = layout + [q for q in range(isa_circuit.num_qubits) if q not in layout]


## Exercise 1 — Box the circuit and learn its noise

<div class="alert alert-block alert-success">

Everything downstream needs a noise model per unique 2-qubit layer: box the circuit, list
the unique layers, generate the candidate error Paulis, and compute the shared,
observable-independent backward bounds (all 20 sites reuse them). This is exactly the SLC
workflow you already ran in QGSS 2026 Lab 3 (sections 2.3–2.4 and 3.2) and the
[SLC tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/pec-with-shaded-lightcones),
so the boxing configuration and the noise-learning budget are **pre-filled** here.

You only need to connect the data flow — fill in the two `...`:

- **Which circuit do we box?** `boxes_pm.run(...)` takes the transpiled ISA circuit.
- **What do we learn the noise model on?** `generate_noise_model_paulis(...)` takes the
  unique 2-qubit layers, not the whole circuit.

The grader checks that the boxed circuit spans all 20 sites and that there is one noise model
per unique layer.

</div>

In [ ]:
# YOUR CODE - you only fill the 2 `...` in the FILL IN block below.
# Everything else in this cell is pre-filled (same SLC workflow as QGSS Lab 3).

# --- pre-filled setup (run as-is) ---
# CPU-side SLC knobs (shared by Ex 1 and Ex 2)
slc_evolution_max_terms = 1000
slc_timeout = 60

# Boxing config: individual_modification (each box separately addressable), noise injected
# on gates only (TREX handles readout in Ex 4), active twirling, all measurements annotated.
boxes_pm = generate_boxing_pass_manager(
    twirling_strategy="active",
    inject_noise_strategy="individual_modification",
    inject_noise_site="after",
    inject_noise_targets="gates",
    measure_annotations="all",
)

# ================== FILL IN (2 blanks) - replace each ... ==================
# 1. Box the circuit.
boxed_circuit = boxes_pm.run(
    ...   # <-- FILL 1/2: which circuit do we box?  (hint: the transpiled ISA circuit)
)

# The unique 2-qubit gate layers that noise learning / injection target (pre-filled).
unique_2q_instructions = find_unique_box_instructions(
    boxed_circuit, normalize_annotations=None, undress_boxes=True
)

# 2. Learn candidate weight-1/2 Pauli error terms on the unique layers.
noise_model_paulis = generate_noise_model_paulis(
    ...,  # <-- FILL 2/2: learn the noise model on what?  (hint: the unique 2-qubit layers)
    backend.coupling_map,
    boxed_circuit,
)
# ============================ end FILL IN ============================

# --- pre-filled: shared backward bounds + noise-learning budget (run as-is) ---
# Shared, observable-independent backward bounds (computed once, reused by every site).
backward_bounds = compute_backward_bounds(
    boxed_circuit,
    noise_model_paulis,
    evolution_max_terms=slc_evolution_max_terms,
    timeout=slc_timeout,
)

noise_learner_options = {
    "num_randomizations": 64,
    "shots_per_randomization": 128,
    "layer_pair_depths": [1, 2, 4, 8, 16],
    "post_selection": {"enable": True, "strategy": "edge", "x_pulse_type": "rx"},
}


Check your setup (no QPU job is submitted):

In [ ]:
# grading — Exercise 1 (local checks only; no QPU job is submitted)
grade_magnetization_ex1(
    boxed_circuit,
    unique_2q_instructions,
    noise_model_paulis,
)

__What did boxing produce?__  Noise learning runs only on the unique layers. So wherever a layer
repeats, the learned model is reused.

In [ ]:
# --- what boxing produced ---
num_boxes = sum(1 for inst in boxed_circuit.data if inst.operation.name == "box")
num_candidate_paulis = sum(p.num_terms for p in noise_model_paulis.values())
print(f"boxes in the circuit       : {num_boxes}")
print(f"unique 2Q layers to learn  : {len(unique_2q_instructions)}")
print(f"unique noise models        : {len(noise_model_paulis)}")
print(f"candidate noise Paulis     : {num_candidate_paulis}")


__Visualize the backward bounds__, split by Pauli type. Darker means a larger bound on the
error's commutator with the initial state $|0\dots0\rangle$. These bounds don't depend on
the observable, so all sites share them. In the $Z$ panel the region near the start stays
pale: the $Z$ errors initially commute with the $|0\rangle$ initial state.

In [ ]:
# --- visualize the backward bounds on the circuit ---
print("backward bounds:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit,
            backward_bounds,
            noise_model_paulis,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            wire_order=wire_order,
            measure_arrows=False,
        )
    )


Forward bounds depend on the observable, so we plot a single bulk site here and the full per-site sweep is Exercise 2. The bound is largest at the measured qubit and spreads back into the circuit from there. Backward bounds are tight near the start and forward bounds near the measurement. Exercise 2 merges the two, and you will see there which one actually wins for a single-site observable. (Layers not reached before the timeout keep the trivial bound of 2.)

In [ ]:
# --- visualize the forward bounds for one representative site (demo only) ---
demo_site = num_qubits // 2   # a bulk site
demo_obs = site_observables[demo_site].apply_layout(layout, num_qubits=isa_circuit.num_qubits)
demo_fwd = compute_forward_bounds(
    boxed_circuit, noise_model_paulis, demo_obs,
    evolution_max_terms=slc_evolution_max_terms,
    eigval_max_qubits=18, atol=1e-8,
    timeout=slc_timeout,
)

print(f"forward bounds (site {demo_site}, Z_{demo_site}):")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit,
            demo_fwd,
            noise_model_paulis,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            wire_order=wire_order,
            measure_arrows=False,
        )
    )


Submit the noise-learning job; it runs on the unique layers and takes about 3 minutes of QPU time. The learned model is a property of the layers, so all sites share it. If you
already have a finished job, paste its id below instead of resubmitting.

In [ ]:
# QPU estimate: ~3 min

NOISE_LEARN_JOB_ID = None   # paste your finished job id here to reuse it
SUBMIT_NOISE_LEARN = True   # set False to skip submitting

noise_learner = NoiseLearnerV3(mode=backend, options=noise_learner_options)

if NOISE_LEARN_JOB_ID is not None:
    nl_job = service.job(NOISE_LEARN_JOB_ID)
    print(f"Reusing saved job: {NOISE_LEARN_JOB_ID}")
elif SUBMIT_NOISE_LEARN:
    nl_job = noise_learner.run(unique_2q_instructions)   # ~3 min of QPU time
    NOISE_LEARN_JOB_ID = nl_job.job_id()
    print(f"Submitted noise-learning job: {NOISE_LEARN_JOB_ID}")
else:
    print("Set SUBMIT_NOISE_LEARN=True to submit, or paste a saved job id.")


In [ ]:
print("noise-learning job status:", nl_job.status())

In [ ]:
assert nl_job.backend().name == backend.name, (
    f"this job ran on {nl_job.backend().name}, but the session backend is {backend.name}. "
    "Reusing it would mix noise data across devices; rerun from the backend cell on the "
    "job's backend, or submit a fresh job.")
noise_learner_result = nl_job.result()
refs_2_plm = noise_learner_result.to_dict(unique_2q_instructions, require_refs=False)
id_map = map_modifier_ref_to_ref(boxed_circuit)
print(f"learned noise for {len(refs_2_plm)} layer(s)")


## Exercise 2 — Per-site SLC bounds

<div class="alert alert-block alert-success">

The backward bounds from Exercise 1 are observable-*independent* and shared by every site.
Here you add the observable-*dependent* half, per site $i$, and choose which errors to
mitigate. All of the plumbing from QGSS Lab 3 is pre-filled — the speed-limit tightening,
the local-scale budget, and the $\gamma$ conversion — so you only connect the data flow.

Fill in the two `...`:

- **`compute_forward_bounds`** bounds each candidate error's commutator with the observable,
  so it needs *this site's* observable, `isa_observables[i]`.
- **`merge_bounds`** combines this site's forward bounds with the **shared backward bounds**
  from Exercise 1 (`backward_bounds`), switching at the bias-minimizing layer.

`compute_local_scales` (pre-filled: `bias_tolerance = 1e-6`, `sampling_cost_budget = np.inf`)
selects the error terms, and `gamma_from_noisy_boxes` turns the chosen scales into the
per-site $\gamma$ (sampling overhead $\gamma^2$). The
[SLC addon docs](https://qiskit.github.io/qiskit-addon-slc/) and
[arXiv:2409.04401](https://arxiv.org/abs/2409.04401) cover the theory.

The grader checks you have one local-scale result and one $\gamma \ge 1$ per site (all 20).

</div>

In [ ]:
# Per-site SLC bounds (CPU). backward_bounds / noise_model_paulis are shared from Ex 1.
slc_eigval_max_qubits = 18
slc_atol = 1e-8
bias_tolerance = 1e-6   # upper bound on tolerated residual bias

meas_box = boxed_circuit.data[-1]
canonical_qubits = [idx for idx, q in enumerate(boxed_circuit.qubits) if q in meas_box.qubits]

isa_observables, site_bases_canon = [], []
site_reversers = []        # CANONICAL (Z@c(i)) -> trex : 20-wide noise map is canonical-ordered
site_reversers_meas = []   # VIRTUAL   (Z@i)    -> executor : datum["meas"] is clbit/virtual order
for i in range(num_qubits):
    isa_observables.append(
        site_observables[i].apply_layout(layout, num_qubits=isa_circuit.num_qubits)
    )
    obs_canon = map_observable_virtual_to_canonical(site_observables[i], layout, canonical_qubits)
    bases_i, reverser_i = get_measurement_bases(obs_canon)
    site_bases_canon.append(bases_i[0])
    site_reversers.append(reverser_i)                       # canonical -> trex
    _, rev_meas = get_measurement_bases(site_observables[i])
    site_reversers_meas.append(rev_meas)                    # virtual -> executor


In [ ]:
# YOUR CODE - you only fill the 2 `...` in the FILL IN block inside the loop.
# Everything else is pre-filled.

local_scales_per_site, gamma_per_site = [], []
fwd_per_site, merged_per_site = [], []   # kept for the merged-bounds view below
for i in range(num_qubits):
    # ============ FILL IN (2 blanks, per site) - replace each ... ============
    this_site_observable   = ...  # <-- FILL 1/2: this site's observable  (hint: isa_observables[i])
    shared_backward_bounds = ...  # <-- FILL 2/2: the shared backward bounds from Ex 1  (hint: backward_bounds)
    # ============================== end FILL IN ==============================

    # --- pre-filled below (run as-is) ---
    # 1. Forward bounds for this site (observable-DEPENDENT).
    fwd = compute_forward_bounds(
        boxed_circuit, noise_model_paulis,
        this_site_observable,
        evolution_max_terms=slc_evolution_max_terms,
        eigval_max_qubits=slc_eigval_max_qubits, atol=slc_atol,
        timeout=slc_timeout,
    )
    # Tighten the forward bounds with the quantum speed limit (provided).
    fwd = tighten_with_speed_limit(fwd, boxed_circuit, noise_model_paulis, this_site_observable)
    # 2. Merge tightened forward + shared backward bounds, using the learned rates refs_2_plm.
    merged_i = merge_bounds(boxed_circuit, fwd, shared_backward_bounds, refs_2_plm)
    # 3. Select which errors to mitigate (bias_tolerance caps the residual bias).
    ls_i, cost_i, bias_i = compute_local_scales(
        boxed_circuit, merged_i, refs_2_plm,
        sampling_cost_budget=np.inf, bias_tolerance=bias_tolerance,
    )
    fwd_per_site.append(fwd)
    merged_per_site.append(merged_i)
    local_scales_per_site.append(ls_i)
    gamma_per_site.append(gamma_from_noisy_boxes(refs_2_plm, id_map, ls_i))


Check the per-site bounds:

In [ ]:
# grading — Exercise 2 (per-site SLC bounds)
grade_magnetization_ex2(gamma_per_site)

__Where does backward end and forward begin?__ `merge_bounds` picks one box index: every box before it uses the backward bound, every box from there on uses the forward bound. The switch point is chosen to minimize the total bias, weighted by the learned noise rates. The cell below prints that index for the middle site, draws the merged bounds with the same `draw_shaded_lightcone` call as in Exercise 1, and counts how many error terms sit at the maximum bound of 2. Those are terms the classical propagation either never reached before the timeout or found to be fully relevant; the drawing shades both the same way, so the count is the only way to see how many there are.

For a single-site observable $Z_i$ the forward lightcone is narrow, so the forward bound usually wins at every box. Expect the switch at or near box 0 and a merged picture that looks like the forward one. Backward bounds start to matter for observables with a wide lightcone, such as a Z string across the whole chain or a sum over many sites.

__How this relates to the paper.__ The bias bound needs one cut through the circuit, the *spacetime partition* $T$ of [arXiv:2409.04401](https://arxiv.org/abs/2409.04401). Errors before the cut are bounded by how much they disturb the initial state (backward), errors after it by how much they disturb the observable (forward). In the paper $T$ can bend from qubit to qubit, which gives the curved boundary in its Fig. 3. The add-on does not implement that per-qubit partition yet; it uses the single box index described above, so the add-on cannot reproduce that figure directly.

In [ ]:
# --- merged bounds for the middle site (CPU only, no QPU job) ---
site = num_qubits // 2
fwd_site, merged_site = fwd_per_site[site], merged_per_site[site]

# merge_bounds returns the forward or backward entry itself for each box, so identity
# tells us where the switch happened. id_map lists boxes in circuit order.
box_refs = list(id_map)
switch_idx = next((k for k, ref in enumerate(box_refs) if merged_site[ref] is fwd_site[ref]),
                  len(box_refs))
if switch_idx == 0:
    print(f"site {site}: forward bounds for all {len(box_refs)} boxes (no backward region)")
else:
    print(f"site {site}: backward bounds for boxes 0..{switch_idx - 1}, "
          f"forward bounds for boxes {switch_idx}..{len(box_refs) - 1}")

# error terms sitting at the maximum bound of 2: either never reached by the propagation
# (timeout / truncation) or genuinely maximal. Either way, SLC cannot shade them.
def count_trivial(bounds):
    rates = np.concatenate([bounds[ref].rates for ref in box_refs])
    return int((rates >= 2 - 1e-9).sum()), rates.size

for name, bnd in [("backward", backward_bounds), ("forward", fwd_site), ("merged", merged_site)]:
    n_triv, n_all = count_trivial(bnd)
    print(f"{name:8s}: {n_triv} of {n_all} error terms at the maximum bound of 2")

print(f"merged bounds (site {site}, Z_{site}):")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit,
            merged_site,
            noise_model_paulis,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            wire_order=wire_order,
            measure_arrows=False,
        )
    )


__Sampling overhead per site.__ $\gamma^2$ tells you how much the shot count grows relative
to an ideal run. Bulk sites usually cost more than edge sites: with neighbours on both sides,
their lightcone catches more errors, and those errors have to be mitigated. The red dots mark
the full-PEC baseline, the cost of cancelling every learned error; it doesn't depend on the
observable, so it sits at the same height for all sites.

In [ ]:
# --- per-site sampling overhead, with the full-PEC baseline ---
overhead = np.array(gamma_per_site, dtype=float) ** 2

# Full PEC cancels every learned error (no local scales), so its gamma is a
# property of the whole circuit, not of the observable.
gamma_full = gamma_from_noisy_boxes(refs_2_plm, id_map)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(range(num_qubits), overhead, color="tab:blue", alpha=0.8,
              label="SLC (per site)")
ax.bar_label(bars, labels=[f"{v:.2f}" for v in overhead], padding=2, fontsize=8)
ax.plot(range(num_qubits), [gamma_full**2] * num_qubits, linestyle="none",
        marker="o", ms=4, color="tab:red",
        label=f"full PEC  $\\gamma^2$ = {gamma_full**2:.2f}")
ax.set_xlabel("site index $i$")
ax.set_ylabel(r"sampling overhead  $\gamma^2$")
ax.set_title(r"SLC sampling overhead per site vs full PEC")
ax.set_xticks(range(num_qubits))
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.show()

print(f"full PEC       : gamma^2 = {gamma_full**2:.2f} (site-independent)")
print(f"cheapest site  : i={int(overhead.argmin())}  (gamma^2 = {overhead.min():.2f})")
print(f"costliest site : i={int(overhead.argmax())}  (gamma^2 = {overhead.max():.2f})")


__Trade sampling overhead against residual bias.__ The tutorial sweeps the bias tolerance up to 10% and plots the sampling cost each tolerance buys (its Step 3 figure). Here is the same sweep for the middle site, reusing its merged bounds from Exercise 2. CPU only, no QPU job. On this shallow circuit with a single-site observable, expect a nearly flat curve: there is little cost left to trade away.

In [ ]:
# --- trade sampling overhead against residual bias (one site, CPU only) ---
site = num_qubits // 2
merged_site = merged_per_site[site]

tolerances = [0.0, 0.001] + [0.01 * k for k in range(1, 11)]
biases, costs = [], []
bias_ceiling = 0.0   # best estimate of the site's total bias bound (grows monotonically)
for tol in tolerances:
    try:
        _, cost_t, bias_t = compute_local_scales(
            boxed_circuit, merged_site, refs_2_plm,
            sampling_cost_budget=np.inf, bias_tolerance=tol,
        )
        bias_ceiling = max(bias_ceiling, bias_t)
    except IndexError:
        # Once the tolerance exceeds this site's total bias bound, no error term is
        # worth mitigating: compute_local_scales builds an all-False mask and trips
        # on `samp_cost_accum[mask][-1]`. Physically this point is "mitigate nothing"
        # — no sampling overhead (gamma^2 = 1), and the remaining bias is just the
        # un-mitigated total (~bias_ceiling, the largest remaining bias seen so far).
        cost_t, bias_t = 1.0, bias_ceiling
    biases.append(bias_t)
    costs.append(cost_t)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter([0], [gamma_full**2], marker="D", c="tab:orange", zorder=3, label="full PEC")
ax.plot(100 * np.array(biases), np.array(costs), "o-", c="tab:blue", label="PEC+SLC")
ax.set_xlabel("remaining bias [%]")
ax.set_ylabel(r"sampling overhead  $\gamma^2$")
ax.set_yscale("log")
ax.set_title(f"Bias-cost tradeoff (site {site})")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()

print(f"full PEC gamma^2 = {gamma_full**2:.2f}")
print(f"PEC+SLC  gamma^2 = {costs[0]:.2f} at zero tolerated bias, "
      f"{costs[-1]:.2f} at 10%")


## Exercise 3 — Assemble the randomized `Executor` program

<div class="alert alert-block alert-success">

`samplomatic.build` turns the boxed circuit into a `(template, samplex)` pair, and a
post-selection pass manager (pre-filled) adds the spectator and post-selection measures,
producing `final_template_circuit`. You then append one samplex item per site
(anti-noise on) plus one unmitigated baseline item (anti-noise off) — `num_qubits + 1`
items in a single job.

The binding logic is fully written for you (reading the `noise_scales` refs from the
samplex interface and binding **every** ref — see the warning below). The **baseline item
at the bottom is also complete and works as your template**: the three per-site `...` are
exactly its three arguments.

Fill in the three `...`:

- **`samplomatic.build(...)`** takes the `boxed_circuit`.
- **`circuit=...`** — which of the two template circuits do we run? Not the bare
  `template_circuit`, but `final_template_circuit` (the one with the post-selection measures).
- **`shape=...`** — the randomization shape, `(num_randomizations,)`.

**Watch out:** an unbound `noise_scales.<ref>` does *not* turn that box's noise off —
samplomatic injects its learned noise at full strength. That is why the pre-filled code
binds *every* ref, reading the list from `samplex.inputs().get_specs('noise_scales')`
rather than from `local_scales`.

</div>

In [ ]:
# YOUR CODE - you only fill the 3 `...` in the FILL IN blocks below.
# The binding logic and the baseline item are pre-filled.

num_randomizations = 128       # per site
shots_per_randomization = 64

# ============ FILL IN (blank 1 of 3) - replace ... ============
samplex_source = ...   # <-- FILL 1/3: which circuit do we build the template + samplex from?  (hint: boxed_circuit)
# ==============================================================

# --- pre-filled: build the template, add post-selection, read interface names (run as-is) ---
template_circuit, samplex = samplomatic.build(samplex_source)
post_selection_pm = PassManager([
    AddSpectatorMeasures(backend.coupling_map),
    AddPostSelectionMeasures(x_pulse_type="rx"),
])
final_template_circuit = post_selection_pm.run(template_circuit)

# ============ FILL IN (blanks 2 & 3 of 3) - replace each ... ============
item_circuit = ...   # <-- FILL 2/3: the circuit each per-site item runs  (hint: final_template_circuit, the post-selection template)
item_shape   = ...   # <-- FILL 3/3: the randomization shape               (hint: (num_randomizations,))
# ========================================================================

# --- pre-filled: interface keys, ref binding, program assembly (run as-is) ---
# The basis_changes interface name is assigned by samplex and varies with the circuit.
basis_key = next(
    spec.name.split(".", 1)[1]
    for spec in samplex.inputs().get_specs("basis_changes")
)
# Bind EVERY noise_scales ref explicitly (an unbound ref injects full-strength noise).
noise_scale_refs = [
    spec.name.split(".", 1)[1]
    for spec in samplex.inputs().get_specs("noise_scales")
]

# One samplex item per site (anti-noise on: noise_scales = -1).
program = QuantumProgram(shots=shots_per_randomization, noise_maps=refs_2_plm)
for i in range(num_qubits):
    inp = {f"noise_scales.{ref}": float(-1) for ref in noise_scale_refs}
    inp |= {"basis_changes": {basis_key: site_bases_canon[i]}}
    inp |= {"local_scales": local_scales_per_site[i]}
    program.append_samplex_item(
        circuit=item_circuit,
        samplex=samplex,
        samplex_arguments=samplex.inputs().make_broadcastable().bind(**inp),
        shape=item_shape,
    )

# One unmitigated baseline item (anti-noise off: noise_scales = 0) - pre-filled template.
inp_unmit = {f"noise_scales.{ref}": float(0) for ref in noise_scale_refs}
inp_unmit |= {"basis_changes": {basis_key: site_bases_canon[0]}}
program.append_samplex_item(
    circuit=final_template_circuit,
    samplex=samplex,
    samplex_arguments=samplex.inputs().make_broadcastable().bind(**inp_unmit),
    shape=(num_randomizations,),
)


Check the assembled program:

In [ ]:
# grading — Exercise 3 (assembled Executor program)
grade_magnetization_ex3(program, final_template_circuit, num_qubits, num_randomizations)

Submit the executor job. As before, paste a finished job id below to reuse it instead of
submitting again.

In [ ]:
EXECUTOR_JOB_ID = None   # paste your finished job id here to reuse it
SUBMIT_EXECUTOR = True   # set False to skip submitting

executor = Executor(backend)

if EXECUTOR_JOB_ID is not None:
    job_exec = service.job(EXECUTOR_JOB_ID)
    print(f"Reusing saved job: {EXECUTOR_JOB_ID}")
elif SUBMIT_EXECUTOR:
    job_exec = executor.run(program)   # guardrail: set max_execution_time if desired
    EXECUTOR_JOB_ID = job_exec.job_id()
    print(f"Submitted executor job: {EXECUTOR_JOB_ID}")
else:
    print("Set SUBMIT_EXECUTOR=True to submit, or paste a saved job id.")


In [ ]:
print("executor job status:", job_exec.status())


In [ ]:
results_exec = job_exec.result()

## Exercise 4 — Post-process & assemble the magnetization

<div class="alert alert-block alert-success">

Three corrections recover an unbiased $\langle Z_i\rangle$ from the twirled, noise-injected
bitstrings, and `executor_expectation_values` applies all three at once:

1. **`rescale_factors`** — TREX readout-error rescaling, built by `trex_factors` from the
   learned measurement noise map (pre-computed as `trex` just above the call).
2. **`postselect_mask`** — post-selection that drops shots affected by non-Markovian noise;
   `PostSelector.compute_mask` returns it as `mask` above the call.
3. **`gamma_factor`** — the per-site $\gamma$ renormalization from Exercise 2, i.e.
   `gamma_per_site[i]`, compensating the attenuation from the mitigated errors.

Fill in these three arguments in the **SLC** call. The **raw** curve (pre-filled) uses the
unmitigated baseline item with none of the three corrections and `gamma_factor = 1.0`.

**Reversers — mind the ordering (already pre-filled, but know why):** the measurement noise
map is indexed in *canonical* qubit order, so `trex_factors` takes `site_reversers[i]`
(canonical). But the sampled bitstrings `datum["meas"]` come back in *virtual* (clbit)
order, so `executor_expectation_values` takes `site_reversers_meas[i]`. Passing the wrong
reverser to either silently scrambles the site labels.

</div>

In [ ]:
# YOUR CODE - you only fill the 3 `...` in the FILL IN block inside the loop
# (the three SLC corrections). Everything else is pre-filled.

measurement_noise_map = noise_learner_result[-1].to_pauli_lindblad_map()
# "node" drops a shot if ANY qubit failed its post-selection flip (strict).
# "edge" drops it only if two NEIGHBOURING qubits both failed (keeps more shots).
# The noise learner used "edge"; try both here and compare.
post_selection_strategy = "node"

datum_unmit = results_exec[num_qubits]   # the unmitigated baseline item
ps = PostSelector.from_circuit(circuit=final_template_circuit,
                               coupling_map=backend.coupling_map)

magnetization = np.zeros(num_qubits)
mag_err = np.zeros(num_qubits)
magnetization_raw = np.zeros(num_qubits)
mag_err_raw = np.zeros(num_qubits)

for i in range(num_qubits):
    datum = results_exec[i]
    # trex looks up the (canonical-ordered) measurement noise map.
    trex = trex_factors(measurement_noise_map, site_reversers[i])
    mask = ps.compute_mask(datum, strategy=post_selection_strategy)

    # ============ FILL IN (3 SLC corrections) - replace each ... ============
    ps_mask      = ...  # <-- FILL 1/3: post-selection mask   (hint: mask, computed just above)
    trex_rescale = ...  # <-- FILL 2/3: TREX readout rescale  (hint: trex, computed just above)
    site_gamma   = ...  # <-- FILL 3/3: SLC gamma factor       (hint: gamma_per_site[i])
    # ========================================================================

    # fully corrected (SLC): supply all three corrections.
    # NOTE: datum["meas"] is in virtual (clbit) order -> use site_reversers_meas.
    val = executor_expectation_values(
        datum["meas"], site_reversers_meas[i], None, avg_axis=0,
        measurement_flips=datum["measurement_flips.meas"],
        pauli_signs=datum.get("pauli_signs", None),
        postselect_mask=ps_mask,
        rescale_factors=trex_rescale,
        gamma_factor=site_gamma,
    )
    magnetization[i], mag_err[i] = val[0][0], np.sqrt(val[0][1])

    # raw: the unmitigated baseline item, no corrections applied.
    # (Its pauli_signs are all positive since no anti-noise was drawn; gamma stays 1.)
    val_raw = executor_expectation_values(
        datum_unmit["meas"], site_reversers_meas[i], None, avg_axis=0,
        measurement_flips=datum_unmit["measurement_flips.meas"],
        pauli_signs=datum_unmit.get("pauli_signs", None),
        gamma_factor=1.0,
    )
    magnetization_raw[i], mag_err_raw[i] = val_raw[0][0], np.sqrt(val_raw[0][1])

print("SLC   <Z_i> =", np.round(magnetization, 3))
print("raw   <Z_i> =", np.round(magnetization_raw, 3))
print("mean magnetization: SLC =", round(float(magnetization.mean()), 3),
      " raw =", round(float(magnetization_raw.mean()), 3))


Compute the exact noiseless profile as a reference. At 20 qubits a $2^{20}$ statevector is
still cheap to simulate:

In [ ]:
# Exact reference: noiseless statevector on the virtual (pre-ISA, unmeasured) circuit
exact_circuit = construct_ising_circuit(num_qubits, num_trotter_steps, rx_angle, barrier=False)
sv = Statevector(exact_circuit)
magnetization_exact = np.array([
    sv.expectation_value(site_observables[i]).real for i in range(num_qubits)
])
mean_exact = magnetization_exact.mean()
print("exact <Z_i>      =", np.round(magnetization_exact, 3))
print("exact mean M     =", round(float(mean_exact), 3))


Check that SLC actually beats raw against the exact reference:

In [ ]:
# grading — Exercise 4 (scored: mitigated magnetization profile)
grade_magnetization_ex4(magnetization, magnetization_raw, magnetization_exact)

__Plot__ the SLC, raw, and exact profiles together.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# exact reference (noiseless)
ax.plot(range(num_qubits), magnetization_exact, marker="s", linestyle="--",
        color="grey", label="exact (noiseless)")
ax.axhline(mean_exact, color="grey", lw=0.8, linestyle=":",
           label=f"exact mean M = {mean_exact:.3f}")

# raw measurement (no post-processing)
ax.errorbar(range(num_qubits), magnetization_raw, yerr=mag_err_raw,
            marker="^", capsize=4, linestyle="-", color="tab:orange", alpha=0.7,
            label="raw (no post-processing)")
ax.axhline(magnetization_raw.mean(), color="tab:orange", lw=0.8, linestyle=":",
           label=f"raw mean M = {magnetization_raw.mean():.3f}")

# SLC-mitigated measurement
ax.errorbar(range(num_qubits), magnetization, yerr=mag_err,
            marker="o", capsize=4, linestyle="-", label="SLC-mitigated")
ax.axhline(magnetization.mean(), color="tab:blue", lw=0.8, linestyle=":",
           label=f"SLC mean M = {magnetization.mean():.3f}")

ax.set_xlabel("site index $i$")
ax.set_ylabel(r"$\langle Z_i\rangle$")
ax.set_title("SLC-mitigated vs raw vs exact magnetization profile (forward Ising, 20 qubits)")
ax.set_xticks(range(num_qubits))
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
# Convergence sweep: reuse the Exercise 4 corrections for this site, subsampling the
# randomization axis (axis 0) to the first k randomizations. Mirrors the exact call
# signature used in Exercise 4 above: trex uses the canonical reverser (site_reversers),
# the executor reads datum["meas"] in virtual order (site_reversers_meas).
datum = results_exec[site]
trex = trex_factors(measurement_noise_map, site_reversers[site])   # canonical -> trex
mask = ps.compute_mask(datum, strategy=post_selection_strategy)
psigns = datum.get("pauli_signs", None)
psigns_raw = datum_unmit.get("pauli_signs", None)

ks = np.unique(np.clip(np.round(np.geomspace(1, num_randomizations, 10)).astype(int),
                       1, num_randomizations))
est, err, est_raw, err_raw = [], [], [], []
for k in ks:
    val = executor_expectation_values(
        datum["meas"][:k], site_reversers_meas[site], None, avg_axis=0,   # VIRTUAL -> executor
        measurement_flips=datum["measurement_flips.meas"][:k],
        pauli_signs=(psigns[:k] if psigns is not None else None),
        postselect_mask=mask[:k],
        rescale_factors=trex,
        gamma_factor=gamma_per_site[site],
    )
    est.append(val[0][0]); err.append(np.sqrt(val[0][1]))
    val_raw = executor_expectation_values(
        datum_unmit["meas"][:k], site_reversers_meas[site], None, avg_axis=0,   # VIRTUAL -> executor
        measurement_flips=datum_unmit["measurement_flips.meas"][:k],
        pauli_signs=(psigns_raw[:k] if psigns_raw is not None else None),
        gamma_factor=1.0,
    )
    est_raw.append(val_raw[0][0]); err_raw.append(np.sqrt(val_raw[0][1]))
est = np.array(est); err = np.array(err)
est_raw = np.array(est_raw); err_raw = np.array(err_raw)

exact_site = magnetization_exact[site]
budget_10pct = 0.1 * abs(exact_site)   # tutorial-style reference band: 10% of exact

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between([0, num_randomizations],
                exact_site - budget_10pct, exact_site + budget_10pct,
                color="red", alpha=0.20, label=f"10% budget (±{budget_10pct:.3f})")
ax.axhline(exact_site, color="black", lw=1, label="exact")
ax.errorbar(ks, est_raw, yerr=err_raw, marker="^", ms=4, capsize=3,
            color="tab:orange", alpha=0.7, linestyle="-", label="raw")
ax.errorbar(ks, est, yerr=err, marker="o", ms=4, capsize=3,
            color="tab:blue", linestyle="-", label="SLC-mitigated")
ax.set_xlabel("number of randomizations")
ax.set_ylabel(rf"$\langle Z_{{{site}}}\rangle$")
ax.set_xlim(0, num_randomizations + 4)
ax.set_title(f"Convergence with randomizations (site {site})")
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
plt.show()


In [ ]:
# Total magnetization: sum over sites.
total_magnetization = magnetization.sum()
total_magnetization_raw = magnetization_raw.sum()
total_magnetization_exact = magnetization_exact.sum()

rel_error = abs(total_magnetization - total_magnetization_exact) / abs(total_magnetization_exact)
rel_error_raw = abs(total_magnetization_raw - total_magnetization_exact) / abs(total_magnetization_exact)
print(f"SLC   total magnetization M = {total_magnetization:.3f}   (rel. error {100 * rel_error:.1f}%)")
print(f"raw   total magnetization M = {total_magnetization_raw:.3f}   (rel. error {100 * rel_error_raw:.1f}%)")
print(f"exact total magnetization M = {total_magnetization_exact:.3f}")


## Useful resources

- [SLC addon documentation](https://qiskit.github.io/qiskit-addon-slc/)
- [PEC with shaded lightcones](https://quantum.cloud.ibm.com/docs/en/tutorials/pec-with-shaded-lightcones) tutorial
- QGSS 2026 Lab 3 ([notebook](https://github.com/qiskit-community/qgss-2026/tree/main/lab-3)) for the full walkthrough of Samplomatic, `NoiseLearnerV3`, the `Executor`, and the PNA / SLC add-ons
<!-- TODO: add the new IBM Quantum Learning module here once it is released -->
- [Lightcone shading for classically accelerated quantum error mitigation](https://arxiv.org/abs/2409.04401) (the SLC paper)
- [PEC with sparse Pauli–Lindblad models](https://arxiv.org/abs/2201.09866) (the noise model SLC builds on)
- [Error mitigation overview](https://quantum.cloud.ibm.com/docs/guides/error-mitigation-overview)
- Next step: **Noise Busters** (`noise_busters.ipynb`), the open challenge that uses this pipeline